# Ray Basics

Ray Core provides a small number of core primitives (i.e., tasks, actors, objects) for building and scaling distributed applications. Below we’ll walk through simple examples that show you how to turn your functions and classes easily into Ray tasks and actors, and how to work with Ray objects.

We need to first import and initialize ray 

In [2]:
import ray

# Let's start Ray
if ray.is_initialized():
    ray.shutdown()
ray.init()

2025-03-13 09:58:15,714	INFO client_builder.py:244 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Python version:,3.9.21
Ray version:,2.43.0
Dashboard:,http://10.10.2.206:8265


(my_task pid=1394929) Starting task with workload 4...
(my_task pid=1394934) Starting task with workload 3...
(my_task pid=1394933) Starting task with workload 2...
(my_task pid=1394930) Starting task with workload 1...
(my_task pid=3259, ip=10.10.2.65) Starting task with workload 5...
(my_task pid=1394930) Task with workload 1 completed.
(my_task pid=1394933) Task with workload 2 completed.
(my_task pid=1394934) Task with workload 3 completed.
(my_task pid=1394929) Task with workload 4 completed.
(my_task pid=3259, ip=10.10.2.65) Task with workload 5 completed.
(task1 pid=1394929) Starting Task1 with workload 1...
(task1 pid=1394934) Starting Task1 with workload 2...
(task1 pid=1394933) Starting Task1 with workload 3...
(task1 pid=1394930) Starting Task1 with workload 4...
(task1 pid=3259, ip=10.10.2.65) Starting Task1 with workload 5...
(task1 pid=1394929) Task1 with workload 1 completed.
(task2 pid=1394929) Starting Task2 with workload 1...
(task2 pid=1394929) Task2 with workload 1 

**Note: to access the dashboard, you need to access 160.85.253.73:8265 from ZHAW network:**

## Tasks

Ray lets you run functions as remote tasks in the cluster. To do this, you decorate your function with @ray.remote to declare that you want to run this function remotely. Then, you call that function with .remote() instead of calling it normally. This remote call returns a future, a so-called Ray object reference, that you can then fetch with ray.get:

In [3]:
# Define the square task.
@ray.remote
def square(x):
    return x * x

# Launch four parallel square tasks.
futures = [square.remote(i) for i in range(4)]

# Retrieve results.
print(ray.get(futures))
# -> [0, 1, 4, 9]

[0, 1, 4, 9]


### But does it really run in parallel?

#### Sequential

In [4]:
import time

def my_task(workload):
    print(f"Starting task with workload {workload}...")
    time.sleep(workload)
    print(f"Task with workload {workload} completed.")
    return workload 

t0=time.time()
results = [my_task(i) for i in range(1, 6)]
t1=time.time()
print(f"Time: {t1-t0}")
print(f"Results: {results}")

Starting task with workload 1...
Task with workload 1 completed.
Starting task with workload 2...
Task with workload 2 completed.
Starting task with workload 3...
Task with workload 3 completed.
Starting task with workload 4...
Task with workload 4 completed.
Starting task with workload 5...
Task with workload 5 completed.
Time: 15.017397165298462
Results: [1, 2, 3, 4, 5]


#### Parallel

In [5]:
import time
@ray.remote
def my_task(workload):
    print(f"Starting task with workload {workload}...")
    time.sleep(workload)
    print(f"Task with workload {workload} completed.")
    return workload 

t0=time.time()
# Launch the my_task function in parallel on the Ray cluster with different workloads
object_ids = [my_task.remote(i) for i in range(1, 6)]

# Retrieve the results of the tasks
results = ray.get(object_ids)
t1=time.time()
print(f"Time: {t1-t0}")
print(f"Results: {results}")

Time: 5.564843654632568
Results: [1, 2, 3, 4, 5]


## Actors

Ray provides actors to allow you to parallelize computation across multiple actor instances. When you instantiate a class that is a Ray actor, Ray will start a remote instance of that class in the cluster. This actor can then execute remote method calls and maintain its own internal state:

In [6]:
# Define the Counter actor.
@ray.remote
class Counter:
    def __init__(self):
        self.i = 0

    def get(self):
        return self.i

    def incr(self, value):
        self.i += value

In [7]:
# Create a Counter actor.
c = Counter.remote()

# Submit calls to the actor. These calls run asynchronously but in
# submission order on the remote actor process.
for _ in range(10):
    c.incr.remote(1)

# Retrieve final actor state.
print(ray.get(c.get.remote()))
# -> 10

10


## Passing an Object

As seen above, Ray stores task and actor call results in its distributed object store, returning object references that can be later retrieved. Object references can also be created explicitly via ray.put, and object references can be passed to tasks as substitutes for argument values:

In [8]:
import numpy as np

# Define a task that sums the values in a matrix.
@ray.remote
def sum_matrix(matrix):
    return np.sum(matrix)

# Call the task with a literal argument value.
print(ray.get(sum_matrix.remote(np.ones((100, 100)))))
# -> 10000.0

# Put a large array into the object store.
matrix_ref = ray.put(np.ones((1000, 1000)))

# Call the task with the object reference as an argument.
print(ray.get(sum_matrix.remote(matrix_ref)))
# -> 1000000.0

10000.0
1000000.0


We can also just pass object references like normal objects. Functions will execute when their inputs become available.

In [9]:
import time

@ray.remote
def task1(workload):
    print(f"Starting Task1 with workload {workload}...")
    time.sleep(workload)
    print(f"Task1 with workload {workload} completed.")
    return workload

@ray.remote
def task2(workload):
    print(f"Starting Task2 with workload {workload}...")
    time.sleep(workload)
    print(f"Task2 with workload {workload} completed.")
    return workload


object_ids1 = [task1.remote(i) for i in range(1, 6)]
object_ids2 = [task2.remote(e) for e in object_ids1]
results = ray.get(object_ids2)
results

[1, 2, 3, 4, 5]

## Shutdown in the end

In [10]:
# shutdown in the end
ray.shutdown()